In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Cargar el dataset original
df = pd.read_csv('data_exam/cars.csv')
df.head()

In [ ]:
df.info()
print("\nEstadísticas descriptivas:")
print(df.describe())

In [ ]:
# 2.1 Detección y eliminación de duplicados reales
print(f"Duplicados detectados: {df.duplicated().sum()}")
df = df.drop_duplicates()

# 2.2 Eliminación de la columna 'Miles' por multicolinealidad perfecta con 'Kms'
if 'Miles' in df.columns:
    df = df.drop(columns=['Miles'])

In [ ]:
# Corrección crítica: Usamos 'Price' como target real y eliminamos cualquier manipulación artificial
X = df.drop(['Price'], axis=1)
y = df['Price']

# Conversión explícita de tipos de datos object a category para modelos nativos
for col in X.select_dtypes(include='object').columns:
    X[col] = X[col].astype('category')

In [ ]:
# División del dataset preservando un conjunto de test independiente
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")

In [ ]:
# Identificar columnas por tipo
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Transformadores individuales
num_transformer = SimpleImputer(strategy='median')
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Procesador global mediante ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

Modelado

In [ ]:
# Modelo Base (Dummy Regressor) envuelto en Pipeline
dummy_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DummyRegressor(strategy='mean'))
])

dummy_cv = cross_validate(dummy_pipe, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(f"Baseline (Dummy) Validation MAE: {-dummy_cv['test_score'].mean().round(2)}")

In [ ]:
# Regresión Lineal envuelta en Pipeline
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_cv = cross_validate(lr_pipe, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(f"Linear Regression Validation MAE: {-lr_cv['test_score'].mean().round(2)}")

In [ ]:
# Random Forest envuelto en Pipeline
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

rf_cv = cross_validate(rf_pipe, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(f"Random Forest Validation MAE: {-rf_cv['test_score'].mean().round(2)}")

In [ ]:
# 1. Identificar de forma explícita qué columnas de X_train son categóricas
# Esto genera una lista de True/False para cada columna, eliminando la fragilidad de 'from_dtype'
is_categorical = [col in X_train.select_dtypes(include=['object', 'category']).columns for col in X_train.columns]

# 2. Instanciar el modelo pasando la máscara booleana directamente
gb_model = HistGradientBoostingRegressor(categorical_features=is_categorical, random_state=42)

# 3. Espacio de búsqueda de hiperparámetros
param_distributions = {
    'max_iter': [50, 100, 150, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, None],
    'min_samples_leaf': [10, 20, 30]
}

# 4. Configurar el RandomizedSearchCV
gb_rs = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1
)

# 5. Ajuste con los datos de entrenamiento
gb_rs.fit(X_train, y_train)

# 6. Mostrar resultados limpios
print(f"Mejores parámetros para HistGradientBoosting: {gb_rs.best_params_}")
print(f"HistGradientBoosting Tuned Validation MAE: {-gb_rs.best_score_:.2f}")

Evaluacion del conjunto de test

In [ ]:
# Utilizar el mejor modelo para predecir sobre el Hold-out set (X_test)
best_model = gb_rs.best_estimator_
predicciones_test = best_model.predict(X_test)

# Calcular el MAE real definitivo del prototipo
mae_final = mean_absolute_error(y_test, predicciones_test)
print(f"El MAE final real medido en el conjunto de prueba (Test) es: {mae_final:.2f}")

# Guardar los resultados en el DataFrame de prueba para exportación
df_testing_results = X_test.copy()
df_testing_results['Real_Price'] = y_test
df_testing_results['Predicted_Price'] = predicciones_test
df_testing_results.to_csv('predicciones_tasacion.csv', index=False)
print("Archivo 'predicciones_tasacion.csv' exportado correctamente.")

5. Conclusiones (1 punto)

- Con qué modelo te quedarías para poner en producción? En caso de que no haya ninguno explica porqué.
- Si tuvieras que mejorar el rendimiento del modelo, cuáles son los siguientes pasos que seguirías?

Conclusión Escrita para defender el examen:
1. Elección del Modelo: Seleccionamos el HistGradientBoostingRegressor optimizado dado que obtiene el menor MAE
   en validación cruzada y mantiene una generalización sólida en el set de prueba independiente (evitando el overfitting).

2. Siguientes pasos para mejora: Realizar ingeniería de características avanzada (ej. ratio de kilómetros por año),
   recolectar variables críticas faltantes (marca, modelo, estado del motor) y probar arquitecturas como XGBoost o LightGBM.